In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# To implement
class DiscretePrior:
    ## Initialise with the beta parameters
    def __init__(self, num_values):
        # initialise uniformly the belief
        self.num_values = num_values
        self.belief = np.ones(num_values) / num_values 
        self.bias_values = np.linspace(0, 1, num_values)

    ## Update the belief when you see a new observation
    def update(self, success):
        if success:
            likelihood = self.bias_values
        else:
            likelihood = 1 - self.bias_values

        self.belief = self.belief * likelihood
        self.belief = self.belief / np.sum(self.belief)

    def mean(self):
        return np.sum(self.bias_values * self.belief)


In [ ]:
# To implement
class BetaConjugatePrior:
    ## Initialise with the beta parameters
    def __init__(self, alpha, beta):
        self.alpha = alpha
        self.beta = beta

    ## Update the belief when you see a new observation
    def update(self, success):
        k = 1 if success else 0
        self.alpha = self.alpha + k
        self.beta = self.beta + (1 - k)

    def mean(self):
        return self.alpha / (self.alpha + self.beta)

In [ ]:
def plot_categorical_distribution(belief):
    x = np.arange(len(belief.bias_values)) / len(belief.bias_values)
    width = 1/len(belief.bias_values)
    
    plt.bar(x + width/2, belief.belief, width=width, alpha=0.8, label="Discrete belief", color="red")


In [ ]:
def plot_beta_distribution(belief, true_param):
    x = np.linspace(0, 1, 20)    
    beta_pdf = stats.beta.pdf(x, belief.alpha, belief.beta)
    # Normalize the Beta PDF so that it sums to 1 over discrete points
    beta_pdf *= np.sum(1 / np.trapz(beta_pdf, x))

    plt.plot(x, beta_pdf, label="Beta belief")
    plt.axvline(x = true_param, color = 'b', linestyle=":", label="True Bias")

In [ ]:
T = 500  # number of time steps
n = 100 # Number of values for the discrete prior
true_bias = 0.6 # probability of heads
alpha_0 = beta_0 = 2 # initial belief parameters
# discrete_belief = DiscretePrior(n)
beta_belief = BetaConjugatePrior(alpha_0, beta_0)

print("Initial beliefs")
plot_categorical_distribution(discrete_belief)
plot_beta_distribution(beta_belief, true_bias)
plt.legend()
plt.show()

for t in range(1, T+1):
    # sample from the n=1 binomial (i.e. Bernoulli) distribution
    # equivalent to performing one coin toss
    is_head = np.random.binomial(1, true_bias) == 1
    ## update the belief
    discrete_belief.update(is_head)
    beta_belief.update(is_head)
    
    if t % 20 == 0:
        print("Tosses:", t+1)
        print("Alpha:", beta_belief.alpha, "Beta:", beta_belief.beta)
        print(f"Beta distribution mean: {beta_belief.mean():.3f}")
        print(f"Discrete distribution mean: {discrete_belief.mean():.3f}")

        plot_categorical_distribution(discrete_belief)
        plot_beta_distribution(beta_belief, true_bias)
        plt.legend()
        plt.show()
